In [15]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

print("Working directory:", Path.cwd())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Working directory: C:\Users\makar\Python\auto-data-scientist


In [16]:
%%writefile src/schemas.py
from pydantic import BaseModel, Field


class PlanStep(BaseModel):
    """One task in the plan. The code agent will carry this out."""
    id: int = Field(description="Step number, starting at 1")
    title: str = Field(description="Short name, e.g. 'Handle missing values'")
    task: str = Field(description="A precise, self-contained instruction for a coder to follow")


class Plan(BaseModel):
    """The full plan the planner agent produces."""
    target_column: str = Field(description="The column being predicted")
    problem_type: str = Field(description="Either 'classification' or 'regression'")
    steps: list[PlanStep]

Overwriting src/schemas.py


In [17]:
%%writefile src/agents/planner.py
import json

from pydantic import ValidationError

from src.llm import chat
from src.schemas import Plan

SYSTEM_PROMPT = """You are a senior data scientist creating a plan for a junior colleague to execute step by step.

RULES
- Reply with ONLY a single JSON object. No explanations, no markdown code fences.
- The JSON must have exactly these keys: "target_column", "problem_type", "steps".
- "problem_type" must be exactly "classification" or "regression".
- "steps" is a list of 4 to 7 objects, each with: "id" (int, starting at 1), "title" (short), "task" (a precise instruction).
- Always include, in this order: (1) explore/profile the data, (2) handle missing values and encode categorical columns,
  (3) engineer or select features if useful, (4) train and compare at least two models, (5) evaluate the best model
  with appropriate metrics and save a chart, (6) save the trained model to disk with joblib.
- Each task must be understandable on its own, without seeing the other steps.
Wrap the JSON in a code block, and add one sentence of explanation before it.
"""


def _extract_json(reply: str) -> str:
    """Pull a JSON object out of a reply that might have extra text around it."""
    reply = reply.strip()
    # Handle models that wrap JSON in ```json ... ``` anyway, despite instructions
    if reply.startswith("```"):
        reply = reply.strip("`")
        if reply.lower().startswith("json"):
            reply = reply[4:]
    # Take the text between the first { and the last }
    start, end = reply.find("{"), reply.rfind("}")
    if start == -1 or end == -1:
        return reply
    return reply[start:end + 1]


def make_plan(goal: str, data_profile: str, max_attempts: int = 3) -> Plan:
    """Ask the LLM for a plan and return it as a validated Plan object.

    Retries if the JSON is malformed or doesn't match the Plan shape,
    feeding the specific error back to the LLM each time.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"GOAL: {goal}\n\nDATA PROFILE:\n{data_profile}"},
    ]

    for attempt in range(1, max_attempts + 1):
        reply = chat(messages)
        json_text = _extract_json(reply)

        try:
            data = json.loads(json_text)             # 1) is it valid JSON at all?
            plan = Plan(**data)                       # 2) does it match our schema?

            # 3) a check Pydantic can't express on its own
            if plan.problem_type not in ("classification", "regression"):
                raise ValueError(f"problem_type must be 'classification' or 'regression', "
                                 f"got '{plan.problem_type}'")
            return plan

        except (json.JSONDecodeError, ValidationError, ValueError) as e:
            print(f"[planner] attempt {attempt}/{max_attempts} failed: {e}")
            if attempt == max_attempts:
                raise
            messages += [
                {"role": "assistant", "content": reply},
                {"role": "user", "content": (
                    f"That was invalid: {e}\n"
                    "Reply again with ONLY a corrected JSON object, no other text."
                )},
            ]

Overwriting src/agents/planner.py


In [18]:
%%writefile -a src/agents/planner.py


def _extract_json(reply: str) -> str:
    """Pull a JSON object out of a reply that might have extra text around it."""
    reply = reply.strip()
    # Handle models that wrap JSON in ```json ... ``` anyway, despite instructions
    if reply.startswith("```"):
        reply = reply.strip("`")
        if reply.lower().startswith("json"):
            reply = reply[4:]
    # Take the text between the first { and the last }
    start, end = reply.find("{"), reply.rfind("}")
    if start == -1 or end == -1:
        return reply
    return reply[start:end + 1]

Appending to src/agents/planner.py


In [19]:
%%writefile -a src/agents/planner.py


def make_plan(goal: str, data_profile: str, max_attempts: int = 3) -> Plan:
    """Ask the LLM for a plan and return it as a validated Plan object.

    Retries if the JSON is malformed or doesn't match the Plan shape,
    feeding the specific error back to the LLM each time.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"GOAL: {goal}\n\nDATA PROFILE:\n{data_profile}"},
    ]

    for attempt in range(1, max_attempts + 1):
        reply = chat(messages)
        json_text = _extract_json(reply)

        try:
            data = json.loads(json_text)             # 1) is it valid JSON at all?
            plan = Plan(**data)                       # 2) does it match our schema?

            # 3) a check Pydantic can't express on its own
            if plan.problem_type not in ("classification", "regression"):
                raise ValueError(f"problem_type must be 'classification' or 'regression', "
                                 f"got '{plan.problem_type}'")
            return plan

        except (json.JSONDecodeError, ValidationError, ValueError) as e:
            print(f"[planner] attempt {attempt}/{max_attempts} failed: {e}")
            if attempt == max_attempts:
                raise
            messages += [
                {"role": "assistant", "content": reply},
                {"role": "user", "content": (
                    f"That was invalid: {e}\n"
                    "Reply again with ONLY a corrected JSON object, no other text."
                )},
            ]

Appending to src/agents/planner.py


In [20]:
from src.config import RUNS_DIR
from src.profiler import quick_profile

ws = RUNS_DIR / "part2_test" / "workspace"   # same folder as Part 2
profile = quick_profile(ws / "data.csv")
print(profile)

Rows: 569, Columns: 31

Columns (name | dtype | missing values | unique values | example):
- mean radius | float64 | 0 | 456 | 17.99
- mean texture | float64 | 0 | 479 | 10.38
- mean perimeter | float64 | 0 | 522 | 122.8
- mean area | float64 | 0 | 539 | 1001.0
- mean smoothness | float64 | 0 | 474 | 0.1184
- mean compactness | float64 | 0 | 537 | 0.2776
- mean concavity | float64 | 0 | 537 | 0.3001
- mean concave points | float64 | 0 | 542 | 0.1471
- mean symmetry | float64 | 0 | 432 | 0.2419
- mean fractal dimension | float64 | 0 | 499 | 0.07871
- radius error | float64 | 0 | 540 | 1.095
- texture error | float64 | 0 | 519 | 0.9053
- perimeter error | float64 | 0 | 533 | 8.589
- area error | float64 | 0 | 528 | 153.4
- smoothness error | float64 | 0 | 547 | 0.006399
- compactness error | float64 | 0 | 541 | 0.04904
- concavity error | float64 | 0 | 533 | 0.05373
- concave points error | float64 | 0 | 507 | 0.01587
- symmetry error | float64 | 0 | 498 | 0.03003
- fractal dimension err

In [21]:
from src.agents.planner import make_plan

plan = make_plan("Predict whether a tumor is malignant or benign, using the 'target' column.", profile)

print("Target:", plan.target_column)
print("Type:", plan.problem_type)
for step in plan.steps:
    print(f"\n{step.id}. {step.title}\n   {step.task}")

Target: target
Type: classification

1. Data Exploration
   Load the dataset, compute summary statistics, visualize distributions of each feature, and examine class balance of the target column.

2. Missing Values & Encoding
   Check for missing values (none expected) and confirm all features are numeric; if any categorical columns existed, apply one-hot encoding, otherwise skip.

3. Feature Engineering/Selection
   Standardize all numeric features using StandardScaler and optionally apply PCA to retain 95% variance or use feature importance from a baseline model to select top features.

4. Model Training & Comparison
   Split data into train/test (e.g., 80/20 stratified), train at least two classifiers such as Logistic Regression and RandomForest, and record cross-validated accuracy.

5. Evaluation & Visualization
   Select the best model based on validation accuracy, compute classification report, ROC AUC, plot ROC curve and confusion matrix, and save the plot as an image file.

6. M

In [23]:
from sklearn.datasets import fetch_california_housing
reg_ws = RUNS_DIR / "part3_regression" / "workspace"
reg_ws.mkdir(parents=True, exist_ok=True)
fetch_california_housing(as_frame=True).frame.to_csv(reg_ws / "data.csv", index=False)

reg_profile = quick_profile(reg_ws / "data.csv")
reg_plan = make_plan("Predict the median house value.", reg_profile)
print(reg_plan.problem_type, "-", reg_plan.target_column)

regression - MedHouseVal


In [24]:
from src.agents.planner import make_plan

plan = make_plan("Predict whether a tumor is malignant or benign, using the 'target' column.", profile)
print("Target:", plan.target_column, "| Type:", plan.problem_type)

Target: target | Type: classification


In [25]:
print(reg_plan.problem_type, "-", reg_plan.target_column)
for step in reg_plan.steps:
    print(f"\n{step.id}. {step.title}\n   {step.task}")

regression - MedHouseVal

1. Data Exploration
   Load the dataset into a pandas DataFrame, generate summary statistics, visualize distributions of each feature and the target, and compute pairwise correlation matrix.

2. Missing Values & Encoding
   Verify that there are no missing values; confirm all columns are numeric and, if any categorical columns appear, apply appropriate encoding such as one‑hot encoding.

3. Feature Engineering/Selection
   Create additional features (e.g., rooms_per_household = AveRooms / AveOccup, bedrooms_per_room = AveBedrms / AveRooms), optionally drop redundant columns, and use correlation or feature‑importance analysis to retain the most predictive features.

4. Model Training & Comparison
   Split the data into training and test sets (e.g., 80/20), standardize numeric features, train at least two regression models such as LinearRegression and RandomForestRegressor, and compute cross‑validated RMSE for each.

5. Best Model Evaluation & Chart
   Select th